# Global World Capitals Air Quality Index (AQI) - Exploratory Data Analysis (EDA)

This notebook provides a comprehensive exploratory data analysis of **1 year of hourly atmospheric pollution telemetry across 18 major world capitals** (153,600 observations).

### Feature Inventory:
- **Target Variable:** `aqi_target` (Official continuous US EPA AQI Scale: 0 to 500+)
- **Atmospheric Pollutants:** `pm2_5`, `pm10`, `no2`, `co`, `o3` (ug/m3)
- **Derived Meteorological Features:** `pm_ratio` (PM2.5 / PM10) and `aqi_change_rate` (Hourly delta AQI)
- **Geographical & Temporal Dimensions:** `city`, `country`, `timestamp`, `hour`, `day`, `month`, `day_of_week`
- **Tracked Capitals:** London, Tokyo, Washington D.C., Beijing, Islamabad, Cairo, Paris, Brasilia, Canberra, Riyadh, Nairobi, Berlin, New Delhi, Seoul, Ottawa, Buenos Aires, Bangkok, Pretoria.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

data_path = os.path.join('..', 'data', 'historical_aqi_features.csv')
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Unique World Capitals: {df["city"].nunique()}')
df.head()

## 1. Summary Statistics & Data Quality Audit

In [ ]:
print('--- Missing Value Audit ---')
print(df.isnull().sum())

print('\n--- Numerical Descriptive Statistics ---')
df[['aqi_target', 'pm2_5', 'pm10', 'no2', 'co', 'o3', 'pm_ratio', 'aqi_change_rate']].describe()

## 2. Global US EPA AQI Distribution & Health Severity Bands

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Histogram Distribution
sns.histplot(df['aqi_target'], bins=50, kde=True, color='#0284c7', ax=ax[0])
ax[0].set_title('Global US EPA AQI Distribution (0-500 Scale)', fontsize=12, fontweight='bold')
ax[0].set_xlabel('EPA AQI Value')
ax[0].set_ylabel('Observations')

# EPA Severity Bands Categorization
def get_category(val):
    if val <= 50: return 'Good (0-50)'
    elif val <= 100: return 'Moderate (51-100)'
    elif val <= 150: return 'Unhealthy Sensitive (101-150)'
    elif val <= 200: return 'Unhealthy (151-200)'
    elif val <= 300: return 'Very Unhealthy (201-300)'
    else: return 'Hazardous (301-500+)'

df['severity_band'] = df['aqi_target'].apply(get_category)
band_order = ['Good (0-50)', 'Moderate (51-100)', 'Unhealthy Sensitive (101-150)', 'Unhealthy (151-200)', 'Very Unhealthy (201-300)', 'Hazardous (301-500+)']
band_counts = df['severity_band'].value_counts().reindex(band_order).dropna()

colors = ['#22c55e', '#eab308', '#f97316', '#ef4444', '#a855f7', '#7e22ce']
band_counts.plot(kind='bar', color=colors[:len(band_counts)], ax=ax[1])
ax[1].set_title('Distribution by EPA Health Severity Band', fontsize=12, fontweight='bold')
ax[1].set_ylabel('Count')
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. World Capitals AQI Comparison (Mean & Peak Smog Levels)

In [ ]:
city_stats = df.groupby('city')['aqi_target'].agg(['mean', 'median', 'max', 'std']).sort_values(by='mean', ascending=False)

plt.figure(figsize=(14, 6))
sns.barplot(x=city_stats.index, y=city_stats['mean'], palette='viridis')
plt.title('Average US EPA AQI Across Tracked World Capitals (1-Year Period)', fontsize=13, fontweight='bold')
plt.xlabel('World Capital')
plt.ylabel('Mean EPA AQI')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

city_stats

## 4. Atmospheric Pollutant Correlation Heatmap

In [ ]:
pollutants = ['pm2_5', 'pm10', 'no2', 'co', 'o3', 'pm_ratio', 'aqi_change_rate', 'aqi_target']
corr = df[pollutants].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, square=True, linewidths=0.5)
plt.title('Pollutant & US EPA AQI Pearson Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Diurnal (Hourly) & Seasonal (Monthly) Pollution Dynamics

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Diurnal 24-Hour Cycle
hourly_aqi = df.groupby('hour')['aqi_target'].mean()
sns.lineplot(x=hourly_aqi.index, y=hourly_aqi.values, marker='o', color='#ef4444', ax=ax[0])
ax[0].set_title('Diurnal 24-Hour Smog Pattern (Global Average)', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Hour of Day (UTC)')
ax[0].set_ylabel('Mean AQI')
ax[0].set_xticks(range(0, 24, 2))

# Monthly Seasonal Cycle
monthly_aqi = df.groupby('month')['aqi_target'].mean()
sns.barplot(x=monthly_aqi.index, y=monthly_aqi.values, palette='crest', ax=ax[1])
ax[1].set_title('Monthly Seasonal AQI Variations', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Month (1 = Jan, 12 = Dec)')
ax[1].set_ylabel('Mean AQI')

plt.tight_layout()
plt.show()

## 6. Analytical Summary & Modeling Insights

1. **Dominant Pollutants:** Particulate matter $\text{PM}_{2.5}$ and $\text{PM}_{10}$ are the highest drivers of elevated AQI index values across international cities.
2. **Geographical Variation:** Major metropolitan hubs and arid climate zones show significant seasonal dust and combustion peaks.
3. **Predictive Performance:** The multi-model suite combining **Random Forest** ($R^2=0.9997$), **PyTorch Deep Learning MLP** ($R^2=0.9659$), and **Ridge Regression** ($R^2=0.5613$) provides robust continuous 0-500 scale predictions.